# Colab: resumable, Bridge-compatible cyber adapter training

This notebook trains **one cyber-security adapter only**. It does not train or load the coder adapter. It uses the public Hugging Face defensive cybersecurity instruction dataset, so it does not need a Kaggle account or token.

The adapter is trained against the pinned `Qwen/Qwen3-0.6B` checkpoint and targets `q_proj`, `gate_proj`, `up_proj`, and `down_proj`, the shared exact-base, Q/MLP-only Bridge contract. Training outputs, resumable checkpoints, live JSONL/TensorBoard logs, validation metrics, and the final adapter are saved in a cyber-specific Google Drive folder. Re-running with the same `RUN_ID` resumes from the latest complete checkpoint.

A public cyber LoRA is available, but its model card names `unsloth/Qwen3-0.6B` as its base. This run creates a fresh cyber LoRA on the exact `Qwen/Qwen3-0.6B` revision used by the Bridge setup instead of loading an adapter trained against a different base. This is an adapter training run; it does not verify Bridge crop or generation behavior.

**Expected Drive artifacts:** `checkpoint-N/` folders containing model, optimizer, scheduler, and RNG state; `training_progress.jsonl`; TensorBoard events; `training_history.json`; `adapter_metadata_summary.json`; and `final_adapter/` for inference. The notebook also writes `cyber_adapter_bridge_manifest.json` with the exact base revision and adapter path.

In [ ]:
%pip -q install transformers==4.57.6 "peft>=0.15,<1" "datasets>=3.6,<5" accelerate safetensors huggingface_hub tensorboard

In [ ]:
import gc, json, math, os, time
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU (T4 or better) before running this notebook.")

drive.mount('/content/drive')

# Keep RUN_ID unchanged to resume this cyber run. Use a new value for a fresh experiment.
RUN_ID = 'cyber_qwen3_0.6b_q_mlp_r16_v1'
ROOT = Path('/content/drive/MyDrive/cgr_adapter_checkpoints')
OUTPUT_DIR = ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / 'tensorboard'
LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST_JSON = OUTPUT_DIR / 'run_manifest.json'

BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
DATASET_ID = 'Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset'
TARGET_MODULES = ['q_proj', 'gate_proj', 'up_proj', 'down_proj']
LORA_RANK = 16
LORA_ALPHA = 32
MAX_LENGTH = 768
MAX_TRAIN_SAMPLES = None  # Set to e.g. 2000 only for a short pipeline check.
MAX_EVAL_SAMPLES = 1024   # Deterministic held-out subset to keep repeated evals practical.
RESUME_FROM_LATEST = True
SAVE_STEPS = 100
EVAL_STEPS = 500
LOGGING_STEPS = 10

api = HfApi()
if RUN_MANIFEST_JSON.exists():
    saved_manifest = json.loads(RUN_MANIFEST_JSON.read_text(encoding='utf-8'))
    if saved_manifest.get('base_model_id') != BASE_MODEL_ID or saved_manifest.get('dataset_id') != DATASET_ID:
        raise RuntimeError('This RUN_ID is already tied to a different base or dataset; choose a new RUN_ID.')
    # Resume against the exact commits recorded for this run, even if the Hub has moved since.
    BASE_REVISION = saved_manifest['base_revision']
    DATASET_REVISION = saved_manifest['dataset_revision']
else:
    BASE_REVISION = api.model_info(BASE_MODEL_ID).sha
    DATASET_REVISION = api.dataset_info(DATASET_ID).sha
DEVICE_NAME = torch.cuda.get_device_name(0)
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
print('GPU:', DEVICE_NAME, '| Transformers:', transformers.__version__)
print('Base commit:', BASE_REVISION, '| dataset commit:', DATASET_REVISION)
print('Output:', OUTPUT_DIR)

## 1. Load the public defensive-security data

The source dataset is hosted on Hugging Face and contains `system`, `user`, and `assistant` fields. A fixed seed creates a 5% validation split from its available training partition. The validation subset is capped at 1,024 examples by default; change `MAX_EVAL_SAMPLES` to `None` to evaluate on the full holdout. Dataset revision, split sizes, and token counts are saved in the run manifest.

In [ ]:
raw = load_dataset(DATASET_ID, revision=DATASET_REVISION)
if 'train' not in raw:
    raise RuntimeError(f"Expected a train split; available splits: {list(raw.keys())}")
source = raw['train']
train_eval = source.train_test_split(test_size=0.05, seed=42)
train_raw, eval_raw = train_eval['train'], train_eval['test']
if MAX_TRAIN_SAMPLES is not None:
    train_raw = train_raw.select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
if MAX_EVAL_SAMPLES is not None:
    eval_raw = eval_raw.select(range(min(MAX_EVAL_SAMPLES, len(eval_raw))))
print('Rows:', len(train_raw), 'train /', len(eval_raw), 'validation')
print('Fields:', train_raw.column_names)

# Pin the tokenizer to the same exact model revision used by the adapter.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

DEFAULT_SYSTEM = (
    'You are a cybersecurity assistant focused on defensive security, detection, '
    'incident response, secure configuration, and security education. Keep guidance '
    'within authorized and defensive use.'
)

def format_row(row):
    system_text = str(row.get('system') or DEFAULT_SYSTEM).strip()
    user_text = str(row.get('user') or '').strip()
    answer_text = str(row.get('assistant') or '').strip()
    if not user_text or not answer_text:
        return {'text': ''}
    messages = [
        {'role': 'system', 'content': system_text},
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': answer_text},
    ]
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_text = train_raw.map(format_row).filter(lambda row: bool(row['text']))
eval_text = eval_raw.map(format_row).filter(lambda row: bool(row['text']))
if len(train_text) == 0 or len(eval_text) == 0:
    raise RuntimeError('Formatting produced an empty train or validation split; check dataset fields.')

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_text.map(tokenize_batch, batched=True, remove_columns=train_text.column_names)
eval_ds = eval_text.map(tokenize_batch, batched=True, remove_columns=eval_text.column_names)
token_count_train = sum(len(ids) for ids in train_ds['input_ids'])
print('Tokenized examples:', len(train_ds), '/', len(eval_ds), '| train tokens:', token_count_train)
print('Examples longer than', MAX_LENGTH, 'tokens are truncated from the end.')

## 2. Build the shared-base, query/MLP-only LoRA

This checks the actual trainable LoRA parameter names after wrapping the base. The run stops if it unexpectedly adds K/V projections. Matching the base revision and module targets makes the adapter a candidate for the Bridge architecture; the separate Bridge notebook remains responsible for cache and generation verification.

In [ ]:
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_REVISION,
    torch_dtype=torch.float16,
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
base.config.use_cache = False
base.gradient_checkpointing_enable()
base.enable_input_require_grads()

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base, peft_config)
model.print_trainable_parameters()
trainable_params, all_params = model.get_nb_trainable_parameters()
actual_lora_modules = sorted({
    name.split('.lora_')[0].split('.')[-1]
    for name, _ in model.named_parameters()
    if '.lora_' in name
})
unexpected = sorted(set(actual_lora_modules) - set(TARGET_MODULES))
if unexpected or {'k_proj', 'v_proj'} & set(actual_lora_modules):
    raise RuntimeError(
        f'Unexpected LoRA targets: {unexpected}; actual={actual_lora_modules}'
    )
print('Verified active LoRA modules:', actual_lora_modules)

## 3. Durable checkpoints, logs, and resume

The trainer writes optimizer/model/scheduler/RNG checkpoints to Drive every 100 optimizer steps and evaluates every 500 steps. JSONL telemetry is flushed on each training log and checkpoint. Checkpoint discovery checks for the adapter weights and training-state files, so an interrupted, incomplete newest checkpoint is skipped when an earlier complete checkpoint exists. Keep the same `RUN_ID` and `RESUME_FROM_LATEST=True` after a disconnect.

In [ ]:
PROGRESS_JSONL = OUTPUT_DIR / 'training_progress.jsonl'
BASELINE_JSON = OUTPUT_DIR / 'baseline_eval.json'
SUMMARY_JSON = OUTPUT_DIR / 'adapter_metadata_summary.json'
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
BRIDGE_MANIFEST = ROOT / 'cyber_adapter_bridge_manifest.json'

run_manifest = {
    'run_id': RUN_ID,
    'specialist': 'cybersecurity_defensive',
    'base_model_id': BASE_MODEL_ID,
    'base_revision': BASE_REVISION,
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'train_examples': len(train_ds),
    'eval_examples': len(eval_ds),
    'train_tokens': token_count_train,
    'max_length': MAX_LENGTH,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'max_eval_samples': MAX_EVAL_SAMPLES,
    'target_modules_requested': TARGET_MODULES,
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'transformers_version': transformers.__version__,
    'torch_version': torch.__version__,
    'device_name': DEVICE_NAME,
    'run_started_utc': RUN_STARTED_UTC,
    'save_steps': SAVE_STEPS,
    'eval_steps': EVAL_STEPS,
    'logging_steps': LOGGING_STEPS,
}
if RUN_MANIFEST_JSON.exists():
    previous = json.loads(RUN_MANIFEST_JSON.read_text(encoding='utf-8'))
    stable_keys = [
        'base_model_id', 'base_revision', 'dataset_id', 'dataset_revision',
        'target_modules_requested', 'lora_rank', 'lora_alpha', 'max_length',
    ]
    changed = [key for key in stable_keys if previous.get(key) != run_manifest.get(key)]
    if changed:
        raise RuntimeError(
            f'RUN_ID already has different {changed}; choose a new RUN_ID to avoid mixing experiments.'
        )
    run_manifest['run_started_utc'] = previous.get('run_started_utc', RUN_STARTED_UTC)
RUN_MANIFEST_JSON.write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

class DriveTelemetryCallback(TrainerCallback):
    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.session_start = time.monotonic()
        self.train_start = None
        self.train_start_step = 0
        self.prior_elapsed = 0.0
        if self.log_path.exists():
            for line in self.log_path.read_text(encoding='utf-8').splitlines():
                try:
                    record = json.loads(line)
                    self.prior_elapsed = max(
                        self.prior_elapsed,
                        float(record.get('cumulative_elapsed_seconds', 0)),
                    )
                except (ValueError, TypeError):
                    continue

    def _write(self, record):
        with self.log_path.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(record, ensure_ascii=False, default=float) + '\n')
            stream.flush()
            os.fsync(stream.fileno())

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.monotonic()
        self.train_start_step = state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        elapsed = time.monotonic() - self.session_start
        train_elapsed = time.monotonic() - self.train_start if self.train_start else None
        observed_steps = max(0, state.global_step - self.train_start_step)
        eta_seconds = None
        if train_elapsed is not None and observed_steps > 0 and state.max_steps > state.global_step:
            eta_seconds = (state.max_steps - state.global_step) * train_elapsed / observed_steps
        self._write({
            'event': 'log',
            'logged_utc': datetime.now(timezone.utc).isoformat(),
            'global_step': state.global_step,
            'max_steps': state.max_steps,
            'epoch': state.epoch,
            'session_elapsed_seconds': elapsed,
            'cumulative_elapsed_seconds': self.prior_elapsed + elapsed,
            'training_session_elapsed_seconds': train_elapsed,
            'estimated_remaining_seconds_at_current_session_rate': eta_seconds,
            'peak_vram_gb': torch.cuda.max_memory_allocated() / (1024 ** 3),
            'metrics': logs or {},
        })
        if eta_seconds is not None:
            print(
                f"[ETA] step {state.global_step}/{state.max_steps}; "
                f"about {eta_seconds / 3600:.2f} h at this session's observed rate"
            )

    def on_save(self, args, state, control, **kwargs):
        checkpoint_path = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        self._write({
            'event': 'checkpoint_saved',
            'logged_utc': datetime.now(timezone.utc).isoformat(),
            'global_step': state.global_step,
            'checkpoint_path': str(checkpoint_path),
        })

def latest_complete_checkpoint(output_dir):
    candidates = sorted(
        Path(output_dir).glob('checkpoint-*'),
        key=lambda path: int(path.name.rsplit('-', 1)[-1])
        if path.name.rsplit('-', 1)[-1].isdigit() else -1,
    )
    for checkpoint in reversed(candidates):
        names = {item.name for item in checkpoint.iterdir()} if checkpoint.is_dir() else set()
        has_adapter = any(name.startswith('adapter_model.') for name in names)
        has_optimizer = any(name in {'optimizer.pt', 'optimizer.safetensors'} for name in names)
        has_scheduler = 'scheduler.pt' in names
        has_rng = any(name.startswith('rng_state') for name in names)
        if 'trainer_state.json' in names and has_adapter and has_optimizer and has_scheduler and has_rng:
            return str(checkpoint)
    return None

latest_checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
if RESUME_FROM_LATEST and latest_checkpoint:
    print('Will resume full trainer state from:', latest_checkpoint)
else:
    discovered = get_last_checkpoint(str(OUTPUT_DIR))
    if discovered:
        print('No complete checkpoint found; the newest checkpoint may have been interrupted:', discovered)
    print('No complete checkpoint to resume; starting at step 0.')

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_ID,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    save_only_model=False,
    load_best_model_at_end=False,
    logging_dir=str(LOG_DIR),
    report_to=['tensorboard'],
    include_tokens_per_second=True,
    include_num_input_tokens_seen=True,
    skip_memory_metrics=False,
    optim='adamw_torch',
    seed=42,
    data_seed=42,
    overwrite_output_dir=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[DriveTelemetryCallback(PROGRESS_JSONL)],
)

## 4. Train, save the final adapter, and write paper-ready telemetry

Initial validation loss/perplexity is stored before training and reused after a resume. The final adapter is saved separately from the larger resumable checkpoints. The summary captures the loss history, learning-rate schedule, validation loss/perplexity, parameter count, checkpoint and adapter sizes, measured device, peak VRAM, training wall time, throughput, and verified target modules.

In [ ]:
if SUMMARY_JSON.exists() and FINAL_ADAPTER_DIR.exists():
    completed = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
    if completed.get('base_revision') != BASE_REVISION or completed.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Existing completed summary has a different pinned revision; use a new RUN_ID.')
    print('This RUN_ID already completed. Existing artifacts:')
    print(json.dumps(completed, indent=2))
else:
    if latest_checkpoint and BASELINE_JSON.exists():
        initial_eval = json.loads(BASELINE_JSON.read_text(encoding='utf-8'))
    else:
        initial_eval = trainer.evaluate()
        initial_eval['eval_perplexity'] = math.exp(min(initial_eval['eval_loss'], 700))
        BASELINE_JSON.write_text(json.dumps(initial_eval, indent=2), encoding='utf-8')

    torch.cuda.reset_peak_memory_stats()
    train_started = time.monotonic()
    try:
        train_result = trainer.train(
            resume_from_checkpoint=latest_checkpoint
            if (RESUME_FROM_LATEST and latest_checkpoint)
            else None
        )
    except Exception as exc:
        failure = {
            'event': 'training_exception',
            'utc': datetime.now(timezone.utc).isoformat(),
            'error': repr(exc),
            'latest_complete_checkpoint': latest_complete_checkpoint(OUTPUT_DIR),
        }
        (OUTPUT_DIR / 'last_failure.json').write_text(json.dumps(failure, indent=2), encoding='utf-8')
        raise
    train_wall_seconds = time.monotonic() - train_started

    model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    final_eval = trainer.evaluate()
    final_eval['eval_perplexity'] = math.exp(min(final_eval['eval_loss'], 700))
    final_eval['eval_loss_delta_from_baseline'] = final_eval['eval_loss'] - initial_eval['eval_loss']

    history = trainer.state.log_history
    training_logs = [row for row in history if 'loss' in row and 'eval_loss' not in row]
    eval_logs = [row for row in history if 'eval_loss' in row]
    for row in eval_logs:
        row['eval_perplexity'] = math.exp(min(row['eval_loss'], 700))
    initial_train_loss = training_logs[0]['loss'] if training_logs else None
    final_train_loss = training_logs[-1]['loss'] if training_logs else None

    def directory_size_bytes(path):
        path = Path(path)
        return sum(item.stat().st_size for item in path.rglob('*') if item.is_file()) if path.exists() else 0

    checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
    checkpoint_bytes = directory_size_bytes(checkpoint) if checkpoint else 0
    final_adapter_bytes = directory_size_bytes(FINAL_ADAPTER_DIR)
    trainable_params, total_params = model.get_nb_trainable_parameters()
    observed_modules = sorted({
        name.split('.lora_')[0].split('.')[-1]
        for name, _ in model.named_parameters()
        if '.lora_' in name
    })
    param_summary = {
        'trainable_parameters': int(trainable_params),
        'total_parameters': int(total_params),
        'trainable_percent': 100.0 * trainable_params / total_params,
        'target_modules_requested': TARGET_MODULES,
        'target_modules_observed': observed_modules,
        'k_proj_or_v_proj_adapted': bool({'k_proj', 'v_proj'} & set(observed_modules)),
    }
    training_history = {
        'initial_eval': initial_eval,
        'final_eval': final_eval,
        'trainer_log_history': history,
        'drive_progress_jsonl': str(PROGRESS_JSONL),
    }
    (OUTPUT_DIR / 'training_history.json').write_text(
        json.dumps(training_history, indent=2, default=float), encoding='utf-8'
    )

    progress_rows = []
    if PROGRESS_JSONL.exists():
        for line in PROGRESS_JSONL.read_text(encoding='utf-8').splitlines():
            try:
                progress_rows.append(json.loads(line))
            except ValueError:
                continue
    cumulative_wall_seconds = max(
        [0.0] + [float(row.get('cumulative_elapsed_seconds', 0)) for row in progress_rows]
    )
    hardware = {
        'device_name': torch.cuda.get_device_name(0),
        'peak_vram_gb': torch.cuda.max_memory_allocated() / (1024 ** 3),
        'training_session_wall_clock_seconds': train_wall_seconds,
        'cumulative_logged_training_seconds': cumulative_wall_seconds,
        'train_samples_per_second': train_result.metrics.get('train_samples_per_second'),
        'train_tokens_per_second': train_result.metrics.get('train_tokens_per_second'),
    }
    summary = {
        'run_id': RUN_ID,
        'specialist': 'cybersecurity_defensive',
        'base_model_id': BASE_MODEL_ID,
        'base_revision': BASE_REVISION,
        'dataset_id': DATASET_ID,
        'dataset_revision': DATASET_REVISION,
        'initial_logged_train_loss': initial_train_loss,
        'final_logged_train_loss': final_train_loss,
        'initial_eval_loss': initial_eval.get('eval_loss'),
        'initial_eval_perplexity': initial_eval.get('eval_perplexity'),
        'final_eval_loss': final_eval.get('eval_loss'),
        'final_eval_perplexity': final_eval.get('eval_perplexity'),
        'best_logged_eval_loss': min((row['eval_loss'] for row in eval_logs), default=None),
        'global_step': trainer.state.global_step,
        'learning_rate_last_logged': next(
            (row.get('learning_rate') for row in reversed(history) if 'learning_rate' in row), None
        ),
        'parameters': param_summary,
        'hardware_compute': hardware,
        'latest_complete_checkpoint_path': checkpoint,
        'latest_complete_checkpoint_size_mb': checkpoint_bytes / (1024 ** 2),
        'final_adapter_path': str(FINAL_ADAPTER_DIR),
        'final_adapter_size_mb': final_adapter_bytes / (1024 ** 2),
        'saved_utc': datetime.now(timezone.utc).isoformat(),
    }
    SUMMARY_JSON.write_text(json.dumps(summary, indent=2, default=float), encoding='utf-8')

    bridge_manifest = {
        'specialist': 'cybersecurity_defensive',
        'adapter_path': str(FINAL_ADAPTER_DIR),
        'base_model_id': BASE_MODEL_ID,
        'base_revision': BASE_REVISION,
        'target_modules': observed_modules,
        'dataset_id': DATASET_ID,
        'dataset_revision': DATASET_REVISION,
        'adapter_size_mb': final_adapter_bytes / (1024 ** 2),
        'run_id': RUN_ID,
        'training_summary_path': str(SUMMARY_JSON),
    }
    BRIDGE_MANIFEST.write_text(json.dumps(bridge_manifest, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2, default=float))
    print('Final cyber adapter:', FINAL_ADAPTER_DIR)
    print('Bridge manifest:', BRIDGE_MANIFEST)

## 5. Resume and inspect the run

After a Colab disconnect, reconnect to the same Google Drive and rerun from the top with the same `RUN_ID`. The latest complete `checkpoint-N` restores adapter/model weights, optimizer, scheduler, RNG, and trainer step. The progress JSONL is append-only; checkpoint and final-adapter folders are separate.

TensorBoard can be opened with:

```python
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/cgr_adapter_checkpoints/cyber_qwen3_0.6b_q_mlp_r16_v1/tensorboard
```

The output `cyber_adapter_bridge_manifest.json` records where the adapter is saved and the exact base revision it expects. Copy that adapter folder to the runtime used by the Bridge notebook, load it only on the recorded base revision, and run the Bridge verification separately. This notebook does not itself establish cache identity or cyber task quality; use its saved telemetry and a separate held-out task evaluation for those claims.